<a href="https://colab.research.google.com/github/BhumiPatil1507/YM_functionable_code/blob/main/YM_MoE_RidgeStack_Parted_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install packages, import libraries, and mount Google Drive


In [ ]:
# ============================================================
# YM EXPERT MoE + RIDGESTACK 
#
# Workflow:
#   1. Read YM from the base CSV
#   2. Reproducible 80/20 train-test split
#   3. Train four XGBoost experts
#   4. Generate 3-fold out-of-fold (OOF) predictions on training rows
#   5. Train ExtraTrees MoE router and RidgeStack using OOF predictions
#   6. Evaluate all six models on the untouched test set
#   7. Calculate disagreement uncertainty Di and save outputs
# ============================================================

%pip -q install xgboost scikit-learn scipy joblib

import os
import csv
import gc
import json
import time
import warnings

import numpy as np
import pandas as pd
import xgboost as xgb
import joblib

from google.colab import drive
from sklearn.model_selection import train_test_split, KFold
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ------------------------------------------------------------
drive.mount("/content/drive")

# ------------------------------------------------------------


Mounted at /content/drive


## 2. Configure paths and verify input CSV files


In [ ]:
# 2. CONFIGURATION — CHANGE ONLY IF YOUR FILES ARE ELSEWHERE
# ------------------------------------------------------------
PROJECT_DIR = "/content/drive/MyDrive/Colab"
DATA_FILE = os.path.join(PROJECT_DIR, "polyOne_aa.csv")
FEATURE_DIR = os.path.join(PROJECT_DIR, "polymer_features")

OUTPUT_DIR = os.path.join(PROJECT_DIR, "YM_MOE_RIDGESTACK_results")
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
PRED_DIR = os.path.join(OUTPUT_DIR, "predictions")
OOF_DIR = os.path.join(OUTPUT_DIR, "oof")
SPLIT_DIR = os.path.join(OUTPUT_DIR, "splits")

for folder in [OUTPUT_DIR, MODEL_DIR, PRED_DIR, OOF_DIR, SPLIT_DIR]:
    os.makedirs(folder, exist_ok=True)

TARGET = "YM"
SEED = 42
TEST_SIZE = 0.20
N_FOLDS = 3
CHUNK_SIZE = 5000

# The validation-based expert training starts with this maximum.
MAX_TREES = 2000
VALIDATION_SIZE = 0.10

EXPERTS = ["fingerprints", "descriptors", "graph", "functional_groups"]

# These names match the Cp notebook provided.
FEATURE_FILES = {
    "fingerprints": os.path.join(FEATURE_DIR, "polyOne_aa_fingerprints.csv"),
    "descriptors": os.path.join(FEATURE_DIR, "polyOne_aa_descriptors.csv"),
    "graph": os.path.join(FEATURE_DIR, "polyOne_aa_graph.csv"),
    "functional_groups": os.path.join(FEATURE_DIR, "polyOne_aa_funcgroups.csv"),
}

# ------------------------------------------------------------
# 3. VERIFY INPUT FILES
# ------------------------------------------------------------
for path in [DATA_FILE, *FEATURE_FILES.values()]:
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Required file not found:\n{path}\n"
            "Check PROJECT_DIR, FEATURE_DIR, and feature CSV filenames."
        )

print("Target:", TARGET)
print("Base CSV:", DATA_FILE)
print("Feature directory:", FEATURE_DIR)
print("Output directory:", OUTPUT_DIR)
print("All required input files found.")

# ------------------------------------------------------------


Target: YM
Base CSV: /content/drive/MyDrive/Colab/polyOne_aa.csv
Feature directory: /content/drive/MyDrive/Colab/polymer_features
Output directory: /content/drive/MyDrive/Colab/YM_MOE_RIDGESTACK_results
All required input files found.


## 3. Read YM and create the reproducible train/test split


In [ ]:
# 4. READ TARGET AND CREATE TRAIN/TEST SPLIT
# ------------------------------------------------------------
target_df = pd.read_csv(DATA_FILE, usecols=[TARGET], low_memory=False)
target_all = pd.to_numeric(target_df[TARGET], errors="coerce").to_numpy(dtype=np.float64)
del target_df
gc.collect()

valid_rows = np.flatnonzero(np.isfinite(target_all))
y = target_all[valid_rows].astype(np.float32)
positions = np.arange(len(y))

if len(y) < 10:
    raise ValueError(f"Only {len(y)} rows have valid {TARGET}; insufficient for training.")

train_idx, test_idx = train_test_split(
    positions,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True
)

y_train = y[train_idx]
y_test = y[test_idx]
train_source_rows = valid_rows[train_idx]
test_source_rows = valid_rows[test_idx]

# Save split positions and original source-row positions for reproducibility.
np.save(os.path.join(SPLIT_DIR, "YM_valid_source_rows.npy"), valid_rows)
np.save(os.path.join(SPLIT_DIR, "YM_train_indices.npy"), train_idx)
np.save(os.path.join(SPLIT_DIR, "YM_test_indices.npy"), test_idx)
np.save(os.path.join(SPLIT_DIR, "YM_train_source_rows.npy"), train_source_rows)
np.save(os.path.join(SPLIT_DIR, "YM_test_source_rows.npy"), test_source_rows)

print(f"\nRows with valid YM: {len(y):,}")
print(f"Train rows: {len(train_idx):,} ({len(train_idx)/len(y):.1%})")
print(f"Test rows:  {len(test_idx):,} ({len(test_idx)/len(y):.1%})")
print(f"YM range: {float(np.min(y)):.6g} to {float(np.max(y)):.6g}")

# ------------------------------------------------------------



Rows with valid YM: 500,000
Train rows: 400,000 (80.0%)
Test rows:  100,000 (20.0%)
YM range: 183.68 to 3774.35


## 4. Identify extracted features and define the feature loader


In [ ]:
# 5. IDENTIFY EXTRACTED FEATURE COLUMNS
#    Feature CSVs must be row-aligned with the base CSV and contain YM.
#    Columns already present in the base CSV are not treated as features.
# ------------------------------------------------------------
with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as f:
    base_columns = next(csv.reader(f))
base_set = set(base_columns)

FEATURE_COLUMNS = {}
for expert, path in FEATURE_FILES.items():
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        header = next(csv.reader(f))

    if TARGET not in header:
        raise ValueError(
            f"'{TARGET}' is missing from {path}. "
            "The feature CSV must be aligned with the base data and include the target column."
        )

    cols = [col for col in header if col not in base_set]
    if not cols:
        raise ValueError(
            f"No extracted feature columns found for '{expert}'. "
            "Check whether FEATURE_DIR points to the feature-extraction outputs."
        )

    FEATURE_COLUMNS[expert] = cols
    print(f"{expert:20s}: {len(cols):,} extracted features")

with open(os.path.join(OUTPUT_DIR, "YM_feature_columns.json"), "w") as f:
    json.dump(FEATURE_COLUMNS, f, indent=2)

# ------------------------------------------------------------
# 6. MEMORY-AWARE FEATURE ROW LOADER
#    Loads only the requested original CSV rows and selected feature columns.
# ------------------------------------------------------------
def load_feature_rows(expert, requested_source_rows):
    path = FEATURE_FILES[expert]
    cols = FEATURE_COLUMNS[expert]
    requested_source_rows = np.asarray(requested_source_rows, dtype=np.int64)

    if len(requested_source_rows) == 0:
        return np.empty((0, len(cols)), dtype=np.float32)

    # Read requested rows in ascending order, then restore requested order.
    order = np.argsort(requested_source_rows)
    sorted_rows = requested_source_rows[order]
    X_sorted = np.empty((len(sorted_rows), len(cols)), dtype=np.float32)

    left = 0
    csv_start = 0

    for chunk in pd.read_csv(
        path, usecols=cols, chunksize=CHUNK_SIZE, low_memory=False
    ):
        csv_end = csv_start + len(chunk)
        right = np.searchsorted(sorted_rows, csv_end, side="left")

        if right > left:
            local_rows = sorted_rows[left:right] - csv_start
            X_sorted[left:right] = chunk.iloc[local_rows].to_numpy(
                dtype=np.float32, copy=True
            )

        left = right
        csv_start = csv_end
        del chunk

        if left >= len(sorted_rows):
            break

    if left != len(sorted_rows):
        raise ValueError(
            f"Could not retrieve all requested rows from {path}. "
            "Check that feature CSV row counts/order match the base CSV."
        )

    inverse = np.empty_like(order)
    inverse[order] = np.arange(len(order))
    X = X_sorted[inverse]

    del X_sorted
    gc.collect()

    if expert == "fingerprints" and not np.isfinite(X).all():
        raise ValueError("Fingerprint feature matrix contains missing/non-finite values.")

    return X

# ------------------------------------------------------------


fingerprints        : 2,048 extracted features
descriptors         : 16 extracted features
graph               : 17 extracted features
functional_groups   : 33 extracted features


## 5. Train the four individual XGBoost experts


In [ ]:
# 7. TRAIN FOUR FINAL EXPERTS
#    A 10% validation subset chooses the number of boosting rounds.
#    Then the expert is refit on the full training split using that
#    selected number of rounds. The test set is not used for fitting.
# ------------------------------------------------------------
final_test_predictions = []
expert_metrics = []
tree_counts = {}

for expert in EXPERTS:
    print("\n" + "=" * 72)
    print("TRAINING FINAL EXPERT:", expert, flush=True)
    started = time.time()

    model_path = os.path.join(MODEL_DIR, f"YM_{expert}_expert.json")
    pred_path = os.path.join(PRED_DIR, f"YM_{expert}_test_predictions.npy")
    metrics_path = os.path.join(MODEL_DIR, f"YM_{expert}_metrics.json")
    imputer_path = os.path.join(MODEL_DIR, f"YM_{expert}_imputer.joblib")

    X_train_raw = load_feature_rows(expert, train_source_rows)
    X_test_raw = load_feature_rows(expert, test_source_rows)

    imputer = None
    if expert != "fingerprints":
        # Fit imputation on the training split only.
        imputer = SimpleImputer(strategy="median")
        X_train = imputer.fit_transform(X_train_raw).astype(np.float32, copy=False)
        X_test = imputer.transform(X_test_raw).astype(np.float32, copy=False)
        joblib.dump(imputer, imputer_path)
    else:
        X_train = X_train_raw
        X_test = X_test_raw

    del X_train_raw, X_test_raw
    gc.collect()

    if not np.isfinite(X_train).all() or not np.isfinite(X_test).all():
        raise ValueError(f"Non-finite feature values remain for expert '{expert}'.")

    inner_train, val_pos = train_test_split(
        np.arange(len(y_train)),
        test_size=VALIDATION_SIZE,
        random_state=SEED,
        shuffle=True
    )

    selector = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=MAX_TREES,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="rmse",
        early_stopping_rounds=100,
        n_jobs=-1,
        random_state=SEED,
        verbosity=0
    )

    selector.fit(
        X_train[inner_train],
        y_train[inner_train],
        eval_set=[(X_train[val_pos], y_train[val_pos])],
        verbose=100
    )

    selected_rounds = int(getattr(selector, "best_iteration", MAX_TREES - 1)) + 1
    hit_tree_limit = selected_rounds >= MAX_TREES

    # Refit using all training rows at the validation-selected number of rounds.
    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=selected_rounds,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        n_jobs=-1,
        random_state=SEED,
        verbosity=0
    )
    model.fit(X_train, y_train, verbose=False)

    pred = model.predict(X_test).astype(np.float64)
    metrics = {
        "target": TARGET,
        "expert": expert,
        "feature_count": len(FEATURE_COLUMNS[expert]),
        "R2": float(r2_score(y_test, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_test, pred))),
        "MAE": float(mean_absolute_error(y_test, pred)),
        "selected_trees": int(selected_rounds),
        "hit_max_tree_limit": bool(hit_tree_limit),
        "training_minutes": float((time.time() - started) / 60.0)
    }

    model.save_model(model_path)
    np.save(pred_path, pred)
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)

    final_test_predictions.append(pred)
    expert_metrics.append(metrics)
    tree_counts[expert] = selected_rounds

    print(json.dumps(metrics, indent=2), flush=True)
    if hit_tree_limit:
        print(
            f"WARNING: {expert} selected the MAX_TREES limit ({MAX_TREES}). "
            "Consider increasing MAX_TREES and rerunning if needed.",
            flush=True
        )

    del X_train, X_test, selector, model, imputer
    gc.collect()

test_pred_matrix = np.column_stack(final_test_predictions).astype(np.float64)
np.save(
    os.path.join(PRED_DIR, "YM_all_expert_test_predictions.npy"),
    test_pred_matrix
)
pd.DataFrame(expert_metrics).to_csv(
    os.path.join(OUTPUT_DIR, "YM_individual_expert_metrics.csv"),
    index=False
)

print("\nSelected tree counts for OOF models:", tree_counts)

# ------------------------------------------------------------



TRAINING FINAL EXPERT: fingerprints
[0]	validation_0-rmse:352.23110
[100]	validation_0-rmse:221.02843
[200]	validation_0-rmse:203.32934
[300]	validation_0-rmse:194.67046
[400]	validation_0-rmse:189.06286
[500]	validation_0-rmse:185.05303
[600]	validation_0-rmse:181.76389
[700]	validation_0-rmse:178.83710
[800]	validation_0-rmse:176.38390
[900]	validation_0-rmse:174.02507
[1000]	validation_0-rmse:172.25137
[1100]	validation_0-rmse:170.47068
[1200]	validation_0-rmse:168.83974
[1300]	validation_0-rmse:167.59224
[1400]	validation_0-rmse:166.22939
[1500]	validation_0-rmse:164.97233
[1600]	validation_0-rmse:163.86420
[1700]	validation_0-rmse:162.81945
[1800]	validation_0-rmse:161.85987
[1900]	validation_0-rmse:161.10843
[1999]	validation_0-rmse:160.27551
{
  "target": "YM",
  "expert": "fingerprints",
  "feature_count": 2048,
  "R2": 0.7989620279668397,
  "RMSE": 160.26581299853905,
  "MAE": 117.69331627855301,
  "selected_trees": 2000,
  "hit_max_tree_limit": true,
  "training_minutes": 92

## 6. Generate out-of-fold (OOF) predictions for stacking


In [ ]:
# 8. GENERATE OOF PREDICTIONS FOR STACKING
#    Each training row is predicted by an expert that did not train on it.
#    Test rows are not involved in creating OOF predictions.
# ------------------------------------------------------------
OOF_PATH = os.path.join(OOF_DIR, "YM_oof_predictions.npy")
OOF_PROGRESS = os.path.join(OOF_DIR, "YM_oof_progress.json")

n_train = len(y_train)
oof = np.lib.format.open_memmap(
    OOF_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(n_train, len(EXPERTS))
)
oof[:] = np.nan
oof.flush()

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
completed = []

for fold, (fit_pos, hold_pos) in enumerate(
    kf.split(np.arange(n_train)), start=1
):
    print(
        f"\nOOF fold {fold}/{N_FOLDS}: "
        f"fit={len(fit_pos):,}, holdout={len(hold_pos):,}",
        flush=True
    )

    fit_source = train_source_rows[fit_pos]
    hold_source = train_source_rows[hold_pos]
    y_fit = y_train[fit_pos]

    for j, expert in enumerate(EXPERTS):
        print(f"  OOF expert: {expert}", flush=True)

        X_fit_raw = load_feature_rows(expert, fit_source)
        X_hold_raw = load_feature_rows(expert, hold_source)

        imp = None
        if expert != "fingerprints":
            # Fit a fresh imputer inside this fold to avoid fold leakage.
            imp = SimpleImputer(strategy="median")
            X_fit = imp.fit_transform(X_fit_raw).astype(np.float32, copy=False)
            X_hold = imp.transform(X_hold_raw).astype(np.float32, copy=False)
        else:
            X_fit = X_fit_raw
            X_hold = X_hold_raw

        del X_fit_raw, X_hold_raw
        gc.collect()

        if not np.isfinite(X_fit).all() or not np.isfinite(X_hold).all():
            raise ValueError(f"Non-finite values found in OOF features for {expert}.")

        model = xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=tree_counts[expert],
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            tree_method="hist",
            n_jobs=-1,
            random_state=SEED + fold,
            verbosity=0
        )

        model.fit(X_fit, y_fit, verbose=False)
        oof[hold_pos, j] = model.predict(X_hold).astype(np.float32)
        oof.flush()

        completed.append(f"{expert}_fold_{fold}")
        with open(OOF_PROGRESS, "w") as f:
            json.dump(
                {
                    "target": TARGET,
                    "n_folds": N_FOLDS,
                    "completed_tasks": completed,
                    "completed_count": len(completed),
                    "total_tasks": N_FOLDS * len(EXPERTS)
                },
                f,
                indent=2
            )

        del X_fit, X_hold, model, imp
        gc.collect()

    del fit_source, hold_source, y_fit
    gc.collect()

oof.flush()

if not np.isfinite(oof).all():
    raise ValueError("OOF predictions contain non-finite values. Do not train stackers.")

print(
    f"\nOOF complete: shape={oof.shape}; "
    f"tasks={len(completed)}/{N_FOLDS * len(EXPERTS)}",
    flush=True
)
del oof
gc.collect()

# ------------------------------------------------------------



OOF fold 1/3: fit=266,666, holdout=133,334
  OOF expert: fingerprints
  OOF expert: descriptors
  OOF expert: graph
  OOF expert: functional_groups

OOF fold 2/3: fit=266,667, holdout=133,333
  OOF expert: fingerprints
  OOF expert: descriptors
  OOF expert: graph
  OOF expert: functional_groups

OOF fold 3/3: fit=266,667, holdout=133,333
  OOF expert: fingerprints
  OOF expert: descriptors
  OOF expert: graph
  OOF expert: functional_groups

OOF complete: shape=(400000, 4); tasks=12/12


0

## 7. Train the ExtraTrees MoE router and RidgeStack


In [ ]:
# 9. TRAIN MoE ROUTER AND RIDGESTACK USING OOF PREDICTIONS
# ------------------------------------------------------------
oof_matrix = np.load(OOF_PATH, mmap_mode="r")

if oof_matrix.shape != (len(y_train), len(EXPERTS)):
    raise ValueError(f"Unexpected OOF shape: {oof_matrix.shape}")
if not np.isfinite(oof_matrix).all():
    raise ValueError("OOF prediction matrix contains non-finite values.")

# Nonlinear MoE-style router over the four expert predictions.
router = ExtraTreesRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=20,
    max_features=1.0,
    n_jobs=-1,
    random_state=SEED
)
router.fit(np.asarray(oof_matrix), y_train)
joblib.dump(router, os.path.join(MODEL_DIR, "YM_MoE_router.joblib"))
moe_pred = router.predict(test_pred_matrix)

# RidgeStack: RidgeCV selects alpha using training OOF predictions only.
ridge_stack = RidgeCV(alphas=np.logspace(-4, 4, 25), cv=5)
ridge_stack.fit(np.asarray(oof_matrix), y_train)
joblib.dump(ridge_stack, os.path.join(MODEL_DIR, "YM_RidgeStack.joblib"))
ridge_pred = ridge_stack.predict(test_pred_matrix)

# ------------------------------------------------------------


## 8. Evaluate models, calculate disagreement Di, and save results


In [ ]:
# 10. TEST METRICS FOR ALL FOUR EXPERTS + BOTH STACKERS
# ------------------------------------------------------------
def metrics_row(name, truth, pred):
    return {
        "Model": name,
        "R2": float(r2_score(truth, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(truth, pred))),
        "MAE": float(mean_absolute_error(truth, pred))
    }

rows = [
    metrics_row(expert, y_test, test_pred_matrix[:, i])
    for i, expert in enumerate(EXPERTS)
]
rows.append(metrics_row("MoE_ExtraTrees", y_test, moe_pred))
rows.append(metrics_row("RidgeStack", y_test, ridge_pred))

metrics_df = pd.DataFrame(rows)
metrics_path = os.path.join(OUTPUT_DIR, "YM_experts_MoE_RidgeStack_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)

# ------------------------------------------------------------
# 11. DISAGREEMENT UNCERTAINTY (Di) AND ERROR CORRELATIONS
#    Di = sample standard deviation across four expert predictions.
#    Di is a disagreement score, not a calibrated prediction interval.
# ------------------------------------------------------------
Di = np.std(test_pred_matrix, axis=1, ddof=1)
moe_abs_error = np.abs(y_test - moe_pred)
ridge_abs_error = np.abs(y_test - ridge_pred)

def safe_corr(a, b):
    if len(a) < 2 or np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

uncertainty_df = pd.DataFrame({
    "Source_Row": test_source_rows,
    "Actual_YM": y_test,
    "MoE_Predicted_YM": moe_pred,
    "MoE_Absolute_Error": moe_abs_error,
    "RidgeStack_Predicted_YM": ridge_pred,
    "RidgeStack_Absolute_Error": ridge_abs_error,
    "Disagreement_Di": Di
})

for i, expert in enumerate(EXPERTS):
    uncertainty_df[f"{expert}_Prediction"] = test_pred_matrix[:, i]

uncertainty_path = os.path.join(
    PRED_DIR, "YM_test_predictions_Di_and_errors.csv"
)
uncertainty_df.to_csv(uncertainty_path, index=False)

summary = {
    "target": TARGET,
    "n_valid": int(len(y)),
    "n_train": int(len(y_train)),
    "n_test": int(len(y_test)),
    "split_seed": SEED,
    "test_size": TEST_SIZE,
    "n_oof_folds": N_FOLDS,
    "selected_tree_counts": tree_counts,
    "ridge_alpha_selected": float(ridge_stack.alpha_),
    "Di_mean": float(np.mean(Di)),
    "Di_median": float(np.median(Di)),
    "Di_p95": float(np.percentile(Di, 95)),
    "corr_Di_MoE_absolute_error": safe_corr(Di, moe_abs_error),
    "corr_Di_RidgeStack_absolute_error": safe_corr(Di, ridge_abs_error)
}

with open(os.path.join(OUTPUT_DIR, "YM_run_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

# ------------------------------------------------------------
# 12. REPORT RESULTS AND OUTPUT LOCATIONS
# ------------------------------------------------------------
print("\n" + "=" * 88)
print("YM: INDIVIDUAL EXPERTS vs MoE vs RIDGESTACK — HELD-OUT TEST SET")
print("=" * 88)
print(metrics_df.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

print("\nRidgeStack selected alpha:", ridge_stack.alpha_)
print("\nDISAGREEMENT UNCERTAINTY")
print("Di = sample standard deviation across the four individual expert predictions")
print(f"Mean Di: {summary['Di_mean']:.6f}")
print(f"Median Di: {summary['Di_median']:.6f}")
print(f"95th percentile Di: {summary['Di_p95']:.6f}")
print(
    "Correlation Di vs MoE absolute error: "
    f"{summary['corr_Di_MoE_absolute_error']:.6f}"
)
print(
    "Correlation Di vs RidgeStack absolute error: "
    f"{summary['corr_Di_RidgeStack_absolute_error']:.6f}"
)

print("\nSaved outputs:")
print("Metrics CSV:       ", metrics_path)
print("Predictions + Di:  ", uncertainty_path)
print("Models directory:  ", MODEL_DIR)
print("OOF predictions:   ", OOF_PATH)
print("Split indices:     ", SPLIT_DIR)
print("Run summary:       ", os.path.join(OUTPUT_DIR, "YM_run_summary.json"))
print("\nDONE.")



YM: INDIVIDUAL EXPERTS vs MoE vs RIDGESTACK — HELD-OUT TEST SET
            Model       R2       RMSE        MAE
     fingerprints 0.798962 160.265813 117.693316
      descriptors 0.563879 236.050650 179.003431
            graph 0.443397 266.670203 203.667064
functional_groups 0.196502 320.401244 238.568459
   MoE_ExtraTrees 0.814018 154.147880 114.460788
       RidgeStack 0.810301 155.680488 115.548211

RidgeStack selected alpha: 10000.0

DISAGREEMENT UNCERTAINTY
Di = sample standard deviation across the four individual expert predictions
Mean Di: 130.497541
Median Di: 111.325993
95th percentile Di: 288.892673
Correlation Di vs MoE absolute error: 0.209635
Correlation Di vs RidgeStack absolute error: 0.228756

Saved outputs:
Metrics CSV:        /content/drive/MyDrive/Colab/YM_MOE_RIDGESTACK_results/YM_experts_MoE_RidgeStack_metrics.csv
Predictions + Di:   /content/drive/MyDrive/Colab/YM_MOE_RIDGESTACK_results/predictions/YM_test_predictions_Di_and_errors.csv
Models directory:   /cont

**9. Install/import packages and mount the same Google Drive**

In [ ]:
%pip -q install rdkit xgboost scikit-learn joblib pandas numpy

import os, json, warnings
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
from google.colab import drive
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, Lipinski, rdFingerprintGenerator, rdMolDescriptors

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.error")
RDLogger.DisableLog("rdApp.warning")

drive.mount("/content/drive")
print("Libraries loaded and Drive mounted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Libraries loaded and Drive mounted.


**10. Set paths and verify that the trained YM models exist**

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/Colab"
MODEL_DIR = os.path.join(PROJECT_DIR, "YM_MOE_RIDGESTACK_results", "models")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "YM_new_polymer_predictions")
FEATURE_COLUMNS_FILE = os.path.join(PROJECT_DIR, "YM_MOE_RIDGESTACK_results", "YM_feature_columns.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXPERTS = ["fingerprints", "descriptors", "graph", "functional_groups"]
EXPERT_TO_KIND = {
    "fingerprints": "fingerprints",
    "descriptors": "descriptors",
    "graph": "graph",
    "functional_groups": "funcgroups",
}

required_files = [
    FEATURE_COLUMNS_FILE,
    os.path.join(MODEL_DIR, "YM_MoE_router.joblib"),
    os.path.join(MODEL_DIR, "YM_RidgeStack.joblib"),
] + [os.path.join(MODEL_DIR, f"YM_{e}_expert.json") for e in EXPERTS]
required_files += [os.path.join(MODEL_DIR, f"YM_{e}_imputer.joblib") for e in EXPERTS if e != "fingerprints"]
missing = [p for p in required_files if not os.path.isfile(p)]
if missing:
    raise FileNotFoundError(
        "Some trained YM files are missing. Run the YM training notebook fully first.\n" + "\n".join(missing)
    )

with open(FEATURE_COLUMNS_FILE, "r") as f:
    FEATURE_COLUMNS = json.load(f)
print("Model files found.")
print("Feature counts:", {e: len(FEATURE_COLUMNS[e]) for e in EXPERTS})
print("Output directory:", OUTPUT_DIR)

Model files found.
Feature counts: {'fingerprints': 2048, 'descriptors': 16, 'graph': 17, 'functional_groups': 33}
Output directory: /content/drive/MyDrive/Colab/YM_new_polymer_predictions


**11. Define the same four feature extractors**

In [ ]:
DESCRIPTOR_FUNCS = {
    "MolWt": Descriptors.MolWt,
    "HeavyAtomCount": Lipinski.HeavyAtomCount,
    "NumHAcceptors": Lipinski.NumHAcceptors,
    "NumHDonors": Lipinski.NumHDonors,
    "MolLogP": Descriptors.MolLogP,
    "TPSA": rdMolDescriptors.CalcTPSA,
    "NumRotatableBonds": Lipinski.NumRotatableBonds,
    "RingCount": Lipinski.RingCount,
    "NumAromaticRings": Lipinski.NumAromaticRings,
    "NumAliphaticRings": Lipinski.NumAliphaticRings,
    "FractionCSP3": rdMolDescriptors.CalcFractionCSP3,
    "LabuteASA": rdMolDescriptors.CalcLabuteASA,
    "BalabanJ": Descriptors.BalabanJ,
    "BertzCT": Descriptors.BertzCT,
    "HallKierAlpha": Descriptors.HallKierAlpha,
    "Chi0v": Descriptors.Chi0v,
}

FUNCTIONAL_GROUP_SMARTS = [
    ("fg_alcohol", "[OX2H][CX4]"),
    ("fg_phenol", "[OX2H]c"),
    ("fg_ether", "[OD2]([#6])[#6]"),
    ("fg_aldehyde", "[CX3H1](=O)[#6]"),
    ("fg_ketone", "[#6][CX3](=O)[#6]"),
    ("fg_carboxylic_acid", "[CX3](=O)[OX2H1]"),
    ("fg_ester", "[CX3](=O)[OX2H0][#6]"),
    ("fg_amide", "[NX3][CX3](=[OX1])[#6]"),
    ("fg_primary_amine", "[NX3;H2;!$(NC=O)]"),
    ("fg_secondary_amine", "[NX3;H1;!$(NC=O)]([#6])[#6]"),
    ("fg_tertiary_amine", "[NX3;H0;!$(NC=O)]([#6])([#6])[#6]"),
    ("fg_imine", "[CX3]=[NX2]"),
    ("fg_nitrile", "[CX2]#N"),
    ("fg_nitro", "[N+](=O)[O-]"),
    ("fg_nitro_alt", "N(=O)=O"),
    ("fg_azo", "[NX2]=[NX2]"),
    ("fg_aromatic_n", "[n]"),
    ("fg_pyridine_like_n", "[nH0]"),
    ("fg_aromatic_oxygen", "[o]"),
    ("fg_aromatic_sulfur", "[s]"),
    ("fg_fluorine", "[F]"),
    ("fg_chlorine", "[Cl]"),
    ("fg_bromine", "[Br]"),
    ("fg_iodine", "[I]"),
    ("fg_alkene", "[CX3]=[CX3]"),
    ("fg_alkyne", "[CX2]#[CX2]"),
    ("fg_aromatic_ring", "a1aaaaa1"),
    ("fg_epoxide_like", "[OX2r3]1[CX4r3][CX4r3]1"),
    ("fg_silicon", "[Si]"),
    ("fg_phosphorus", "[P]"),
    ("fg_sulfonyl", "[SX4](=O)(=O)"),
    ("fg_sulfide", "[#16X2]([#6])[#6]"),
    ("fg_carbonate", "[OX2][CX3](=O)[OX2]"),
]
COMPILED_FG = []
for name, smarts in FUNCTIONAL_GROUP_SMARTS:
    query = Chem.MolFromSmarts(smarts)
    if query is None:
        raise ValueError(f"Invalid SMARTS for {name}: {smarts}")
    COMPILED_FG.append((name, query))

FP_GENERATOR = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=2048, includeChirality=False
)

def extract_polymer_features(smiles):
    """Return all four feature dictionaries for one valid SMILES."""
    if not isinstance(smiles, str) or not smiles.strip():
        raise ValueError("SMILES is empty.")
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"RDKit could not parse SMILES: {smiles}")

    fp = FP_GENERATOR.GetFingerprint(mol)
    fp_array = np.zeros((2048,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, fp_array)
    fingerprint_features = {f"fp_{i}": int(fp_array[i]) for i in range(2048)}

    descriptor_features = {}
    for name, fn in DESCRIPTOR_FUNCS.items():
        try:
            value = float(fn(mol))
            descriptor_features[name] = value if np.isfinite(value) else np.nan
        except Exception:
            descriptor_features[name] = np.nan

    n_atoms = mol.GetNumAtoms()
    n_bonds = mol.GetNumBonds()
    heavy = mol.GetNumHeavyAtoms()
    degrees = np.asarray([atom.GetDegree() for atom in mol.GetAtoms()], dtype=float)
    if not degrees.size:
        degrees = np.array([0.0])
    components = len(Chem.GetMolFrags(mol))
    heavy_in_rings = sum(atom.GetAtomicNum() > 1 and atom.IsInRing() for atom in mol.GetAtoms())
    graph_features = {
        "graph_num_atoms": float(n_atoms),
        "graph_num_bonds": float(n_bonds),
        "graph_num_heavy_atoms": float(heavy),
        "graph_num_wildcards": float(sum(atom.GetAtomicNum() == 0 for atom in mol.GetAtoms())),
        "graph_mean_degree": float(degrees.mean()),
        "graph_max_degree": float(degrees.max()),
        "graph_min_degree": float(degrees.min()),
        "graph_degree_std": float(degrees.std()),
        "graph_num_components": float(components),
        "graph_num_rings": float(rdMolDescriptors.CalcNumRings(mol)),
        "graph_num_aromatic_atoms": float(sum(atom.GetIsAromatic() for atom in mol.GetAtoms())),
        "graph_num_aromatic_bonds": float(sum(bond.GetIsAromatic() for bond in mol.GetBonds())),
        "graph_num_single_bonds": float(sum(bond.GetBondType() == Chem.BondType.SINGLE for bond in mol.GetBonds())),
        "graph_num_double_bonds": float(sum(bond.GetBondType() == Chem.BondType.DOUBLE for bond in mol.GetBonds())),
        "graph_num_triple_bonds": float(sum(bond.GetBondType() == Chem.BondType.TRIPLE for bond in mol.GetBonds())),
        "graph_cycle_rank": float(max(0, n_bonds - n_atoms + components)),
        "graph_fraction_heavy_atoms_in_rings": float(heavy_in_rings / heavy) if heavy else 0.0,
    }

    functional_group_features = {
        name: float(mol.HasSubstructMatch(query)) for name, query in COMPILED_FG
    }
    return {
        "fingerprints": fingerprint_features,
        "descriptors": descriptor_features,
        "graph": graph_features,
        "functional_groups": functional_group_features,
    }

print("Feature extraction functions ready.")
print("Expected feature counts:", {k: len(v) for k, v in FEATURE_COLUMNS.items()})

Feature extraction functions ready.
Expected feature counts: {'fingerprints': 2048, 'descriptors': 16, 'graph': 17, 'functional_groups': 33}


**12. Enter new polymer SMILES**

In [ ]:
# EDIT THIS LIST: add one SMILES string per new polymer.
NEW_POLYMERS = [
    {"polymer_id": "new_polymer_1", "smiles": "[*]Oc1ccc(N=Nc2ccc(-c3cc([*])cc(Br)c3Nc3ccc(N4CCCCC4)cc3)cc2)cc1[N+](=O)[O-]"},
    {"polymer_id": "new_polymer_2", "smiles": "[*]NC1CC(C)(C)CC(C)(c2ccc(-c3c(C)cc(C)c(-c4ccc(-c5ccc(N6C(=O)c7cc(Cl)c(N8C(=O)c9cc([*])c(Cl)cc9C8=O)cc7C6=O)cc5C)cc4)c3C)c(C)c2)C1"},
]

if not NEW_POLYMERS:
    raise ValueError("Add at least one polymer to NEW_POLYMERS.")

for item in NEW_POLYMERS:
    if "smiles" not in item:
        raise ValueError("Each entry must have a 'smiles' key.")
    print(item.get("polymer_id", "polymer"), "=>", item["smiles"])


new_polymer_1 => [*]Oc1ccc(N=Nc2ccc(-c3cc([*])cc(Br)c3Nc3ccc(N4CCCCC4)cc3)cc2)cc1[N+](=O)[O-]
new_polymer_2 => [*]NC1CC(C)(C)CC(C)(c2ccc(-c3c(C)cc(C)c(-c4ccc(-c5ccc(N6C(=O)c7cc(Cl)c(N8C(=O)c9cc([*])c(Cl)cc9C8=O)cc7C6=O)cc5C)cc4)c3C)c(C)c2)C1


**13. Extract features, check feature compatibility, and save the feature CSVs**

In [ ]:
all_feature_dicts = []
valid_items = []
errors = []

for i, item in enumerate(NEW_POLYMERS, start=1):
    polymer_id = str(item.get("polymer_id", f"polymer_{i}"))
    smiles = str(item.get("smiles", ""))
    try:
        feats = extract_polymer_features(smiles)
        # Ensure every feature expected by the trained model exists.
        for expert in EXPERTS:
            missing = [c for c in FEATURE_COLUMNS[expert] if c not in feats[expert]]
            if missing:
                raise KeyError(f"{expert}: missing expected columns: {missing[:10]}")
            if len(FEATURE_COLUMNS[expert]) != {"fingerprints":2048, "descriptors":16, "graph":17, "functional_groups":33}[expert]:
                raise ValueError(f"Unexpected feature count in metadata for {expert}.")
        all_feature_dicts.append(feats)
        valid_items.append({"polymer_id": polymer_id, "smiles": smiles})
        print(f"Extracted features for {polymer_id}")
    except Exception as e:
        errors.append({"polymer_id": polymer_id, "smiles": smiles, "error": str(e)})
        print(f"FAILED {polymer_id}: {e}")

if errors:
    pd.DataFrame(errors).to_csv(os.path.join(OUTPUT_DIR, "YM_feature_extraction_errors.csv"), index=False)
    print("Some SMILES failed; details saved to YM_feature_extraction_errors.csv")
if not valid_items:
    raise ValueError("No valid polymers could be featurized. Check the SMILES errors above.")

# Save each expert's extracted feature matrix for inspection/reuse.
for expert in EXPERTS:
    kind = EXPERT_TO_KIND[expert]
    rows = []
    for item, feats in zip(valid_items, all_feature_dicts):
        row = {"polymer_id": item["polymer_id"], "smiles": item["smiles"]}
        row.update({col: feats[expert][col] for col in FEATURE_COLUMNS[expert]})
        rows.append(row)
    out_path = os.path.join(OUTPUT_DIR, f"YM_new_polymer_features_{kind}.csv")
    pd.DataFrame(rows).to_csv(out_path, index=False)
    print(f"Saved {kind} features: {out_path}")


Extracted features for new_polymer_1
Extracted features for new_polymer_2
Saved fingerprints features: /content/drive/MyDrive/Colab/YM_new_polymer_predictions/YM_new_polymer_features_fingerprints.csv
Saved descriptors features: /content/drive/MyDrive/Colab/YM_new_polymer_predictions/YM_new_polymer_features_descriptors.csv
Saved graph features: /content/drive/MyDrive/Colab/YM_new_polymer_predictions/YM_new_polymer_features_graph.csv
Saved funcgroups features: /content/drive/MyDrive/Colab/YM_new_polymer_predictions/YM_new_polymer_features_funcgroups.csv


**14. Load trained models and predict YM**

In [ ]:
# Load four trained XGBoost experts, saved imputers, and stackers.
models = {}
imputers = {}
for expert in EXPERTS:
    model = xgb.XGBRegressor()
    model.load_model(os.path.join(MODEL_DIR, f"YM_{expert}_expert.json"))
    models[expert] = model
    if expert != "fingerprints":
        imputers[expert] = joblib.load(os.path.join(MODEL_DIR, f"YM_{expert}_imputer.joblib"))

moe_router = joblib.load(os.path.join(MODEL_DIR, "YM_MoE_router.joblib"))
ridge_stack = joblib.load(os.path.join(MODEL_DIR, "YM_RidgeStack.joblib"))

expert_prediction_matrix = np.zeros((len(valid_items), len(EXPERTS)), dtype=float)
for row_i, feats in enumerate(all_feature_dicts):
    for col_i, expert in enumerate(EXPERTS):
        cols = FEATURE_COLUMNS[expert]
        X = pd.DataFrame([[feats[expert][c] for c in cols]], columns=cols).to_numpy(dtype=np.float32)
        if expert != "fingerprints":
            X = imputers[expert].transform(X).astype(np.float32, copy=False)
        pred = float(models[expert].predict(X)[0])
        expert_prediction_matrix[row_i, col_i] = pred

moe_predictions = moe_router.predict(expert_prediction_matrix)
ridge_predictions = ridge_stack.predict(expert_prediction_matrix)
Di = np.std(expert_prediction_matrix, axis=1, ddof=1)

results = pd.DataFrame({
    "polymer_id": [x["polymer_id"] for x in valid_items],
    "smiles": [x["smiles"] for x in valid_items],
    "YM_MoE_prediction": moe_predictions,
    "YM_RidgeStack_prediction": ridge_predictions,
    "expert_disagreement_Di": Di,
})
for j, expert in enumerate(EXPERTS):
    results[f"YM_{expert}_prediction"] = expert_prediction_matrix[:, j]

prediction_path = os.path.join(OUTPUT_DIR, "YM_new_polymer_predictions.csv")
results.to_csv(prediction_path, index=False)
print("\nNEW POLYMER YM PREDICTIONS")
display(results)
print("Saved predictions:", prediction_path)
print("Saved extracted feature CSVs in:", OUTPUT_DIR)


NEW POLYMER YM PREDICTIONS


,polymer_id,smiles,YM_MoE_prediction,YM_RidgeStack_prediction,expert_disagreement_Di,YM_fingerprints_prediction,YM_descriptors_prediction,YM_graph_prediction,YM_functional_groups_prediction
0,new_polymer_1,[*]Oc1ccc(N=Nc2ccc(-c3cc([*])cc(Br)c3Nc3ccc(N4...,2334.665656,2333.294296,87.270986,2327.150146,2201.661377,2118.551270,2183.279053
1,new_polymer_2,[*]NC1CC(C)(C)CC(C)(c2ccc(-c3c(C)cc(C)c(-c4ccc...,2132.494349,2133.761136,52.824121,2127.633789,2079.771484,2135.110596,2020.590454


Saved predictions: /content/drive/MyDrive/Colab/YM_new_polymer_predictions/YM_new_polymer_predictions.csv
Saved extracted feature CSVs in: /content/drive/MyDrive/Colab/YM_new_polymer_predictions
